> **Riverside's deployment problem:**  
> The literary editing assistant is ready to ship. The base model — LLaMA-3-7B — produces suggestions the editors genuinely use. The problem: every author on the team works on an Apple Silicon MacBook with 16 GB of unified memory. Manuscripts are confidential; no cloud, no internet during editing sessions.
>  
> **The constraint in numbers:**  
> - LLaMA-3-7B in bf16: **14 GB**  
> - macOS + other apps running simultaneously: **~3 GB**  
> - Available for model: **13 GB** — 1 GB short before a single token is generated  
>  
> The team has three uncomfortable options: downgrade to a 3B model and accept weaker suggestions, pay for cloud inference and break the offline requirement, or compress the 7B model itself. Compression is the only option that keeps quality *and* offline operation — but "quantize the model" is not a single choice. There are five distinct methods, each buying a different amount of memory at a different quality cost.  
>  
> The question is not *whether* to quantize. It's *which* method, at *which* bit-width, keeps Riverside's editors from noticing the difference.


# Quantization in Depth: From 16 GB to 4 GB Without Breaking the Model

| Part | Concept | Riverside question |
| --- | --- | --- |
| 1 | Quantization basics | How does int8 work? What is the rounding error? |
| 2 | Dynamic PTQ | Does int8 hurt quality on our GPT-2 model? |
| 3 | Static PTQ | Can calibration improve int8 accuracy? |
| 4A | GPTQ | Can Hessian-guided int4 be practical? |
| 4B | AWQ | Can activation statistics protect the channels that matter? |
| 5 | GGUF/llama.cpp | Will it run at acceptable speed on Apple Silicon? |
| Appendix A | QAT | When is retraining against simulated quantization noise worthwhile? |
| Appendix B | NF4 and QLoRA | How does a 4-bit frozen base enable memory-efficient adaptation? |
| 7 | Toy to real bridge | What is the final Riverside MacBook recommendation? |

In [ ]:
import subprocess, sys

# Install torch/numpy/matplotlib/transformers only if missing
for pkg in ["torch", "numpy", "matplotlib", "transformers"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import GPT2LMHeadModel, AutoTokenizer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MACBOOK_VRAM_GB = 16.0  # Riverside constraint

print("Loading GPT-2 for quantization experiments...")
try:
    # Load a real GPT-2 checkpoint to quantize; fall back to reference numbers if offline
    model_fp32 = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    # GPT-2 never defined a pad token, so borrow EOS for padding
    tokenizer.pad_token = tokenizer.eos_token
    MODEL_LOADED = True
    n_params = sum(p.numel() for p in model_fp32.parameters())
    model_fp32_gb = n_params * 4 / 1e9
    print(f"  GPT-2: {n_params/1e6:.0f}M params = {model_fp32_gb:.2f} GB (fp32)")
    print(f"  We use this as Riverside's proxy (smaller than 7B but same architecture)")
# Fall back to representative reference numbers if the model can't be downloaded
except Exception as e:
    print(f"  GPT-2 load failed: {e}")
    MODEL_LOADED = False
    model_fp32_gb = 0.47  # GPT-2 in fp32

print(f"\nRiverside constraint: {MACBOOK_VRAM_GB} GB MacBook RAM")
print(f"  7B model at bf16: ~14 GB → leaves only 2 GB for system")
print(f"  Goal: compress to ≤ 4 GB while preserving editing quality")

---

## Part 1 — Quantization Basics: How Does int8 Work?

> **Before the math:** Your weight tensor contains millions of floats — 0.01847, −0.03221, 0.00491. int8 can only represent 256 distinct values across the entire range. You're about to snap every float to its nearest allowed slot. The gap between where a float lives and the nearest slot is the rounding error — every downstream metric (perplexity, output quality) traces back to how large those gaps are across billions of weights. The formula below describes how to choose the 256 slots and how to measure the gap.

**Before viewing the figure:** look for three things: the finite set of representable levels, the distance from each original value to its nearest level, and the outliers that force the scale to cover a wider range.

![int8 quantization: 256 uniform levels on a number line, with the bell-curve weight distribution concentrated near zero and small coral rounding-error bars](images/quantization-rounding-error.png)

**After viewing it:** the coral gaps are the error budget. More levels make each gap smaller; a wider min-to-max range makes every gap larger. One extreme outlier can therefore reduce resolution for all the ordinary values unless calibration chooses a clipping threshold.

Quantization maps floating-point values to a smaller set of integers:

$$x_{quant} = \text{round}\left(\frac{x - x_{min}}{(x_{max} - x_{min}) / (2^{bits} - 1)}\right)$$

In plain English: we divide the float range $[x_{min}, x_{max}]$ into $2^{bits}$ equally-spaced buckets, then snap each weight to its nearest bucket centre. The `scale` is the bucket width; the `zero_point` shifts the grid so that 0.0 maps exactly to an integer — important so padding tokens don't introduce a numerical artefact.

**Representability first:** the integer code is not the weight itself. The pair `(code, scale, zero_point)` identifies a point on a shared grid. Values between grid points are unrepresentable and must round; values beyond the calibrated range must clip.

**Parameters:**

- `scale` = $(x_{max} - x_{min}) / 255$ for uint8; it is the distance between adjacent representable real values
- `zero_point` = the integer code whose dequantized value is closest to real zero
- `rounding_error` = $x - \widehat{x}$ after mapping to the nearest grid point
- `clipping_error` = the additional error when $x$ lies outside the chosen range

**Toy values:** let the observed range be $[-1, 1]$. Then $s = 2/255 \approx 0.00784$ and $z = \operatorname{round}(1/s) = 128$. For $x=0.30$:

$$
q = \operatorname{round}(0.30 / 0.00784 + 128) = 166
$$

$$
\widehat{x} = (166 - 128)(0.00784) \approx 0.298, \qquad |x-\widehat{x}| \approx 0.002
$$

That error is below $s/2 \approx 0.00392$. But $x=1.20$ lies outside the calibrated range: its code clips at 255, so no extra bit pattern can represent the missing 0.20. This is why calibration and clipping policy matter as much as bit width.

```mermaid
flowchart LR
    X[Float value x] --> R[Choose range or clip]
    R --> Q[Quantize: round x / s + z]
    Q --> I[Store integer code q]
    I --> D[Dequantize: s times q - z]
    D --> XH[Approximation x-hat]
    XH --> E[Measure rounding and clipping error]
```

For int8, 256 levels usually make rounding small. Int4 has only 16 levels, so scale selection, group size, and outlier handling become much more consequential.

**Why does this compress memory?**  
A 32-bit float occupies 4 bytes. An int8 value occupies 1 byte — 4× smaller.  
For a 7B-parameter model: 7 × 10⁹ × 4 bytes = 28 GB (fp32) → 7 GB (int8) → 3.5 GB (int4).

#### #### Predict first

You are about to quantize a real GPT-2 weight tensor (768×768 floats, values mostly in [−0.07, +0.07]) from fp32 to int8, then dequantize back. The **mean absolute error per weight** will be closest to:

**(a)** < 0.00001 — essentially zero; int8 has negligible rounding for small-magnitude weights  
**(b)** ≈ 0.0001–0.0005 — tiny per weight, but scale × billions of weights = measurable signal  
**(c)** ≈ 0.001–0.01 — visible rounding gaps in the weight histogram  

Lock in your answer before running the next cell.


In [ ]:
#  Part 1: Quantization and dequantization
# Map float weights onto the 256 uint8 buckets defined by scale/zero_point
def quantize_int8(tensor):
    x_min, x_max = tensor.min().item(), tensor.max().item()
    scale = (x_max - x_min) / 255.0
    zero_point = round(-x_min / scale)
    q = torch.round(tensor / scale + zero_point).clamp(0, 255).to(torch.uint8)
    return q, scale, zero_point


# Reverse the mapping to reconstruct approximate float values
def dequantize_int8(q_tensor, scale, zero_point):
    return (q_tensor.float() - zero_point) * scale


# Demonstrate on a weight tensor from GPT-2's first layer
# Use a real GPT-2 weight tensor when available, otherwise a synthetic one of the same scale
if MODEL_LOADED:
    # Get a real weight tensor
    weight = list(model_fp32.parameters())[0].detach().cpu()
else:
    torch.manual_seed(42)
    weight = torch.randn(768, 768) * 0.02  # typical LLM weight scale

print(f"Weight tensor: shape={weight.shape}, dtype={weight.dtype}")
print(f"  Range: [{weight.min():.4f}, {weight.max():.4f}]")
print(f"  Std:   {weight.std():.4f}")
print()

# Round-trip the real weight tensor through int8 to measure the damage
q, scale, zp = quantize_int8(weight)
weight_reconstructed = dequantize_int8(q, scale, zp)

# Worst-case and average reconstruction error introduced by quantization
max_abs_error = (weight - weight_reconstructed).abs().max().item()
mean_abs_error = (weight - weight_reconstructed).abs().mean().item()
print(f"After int8 quantization + dequantization:")
print(f"  scale = {scale:.6f}, zero_point = {zp}")
print(f"  Max absolute error:  {max_abs_error:.6f}")
print(f"  Mean absolute error: {mean_abs_error:.6f}")
print(
    f"  Error as % of range: {mean_abs_error / (weight.max()-weight.min()).item() * 100:.3f}%"
)
print()
print(
    f"Memory: fp32 = {weight.numel()*4/1e6:.2f} MB → int8 = {q.numel()*1/1e6:.2f} MB  (4× smaller)"
)

In [ ]:
#  Part 1: Quantization error distribution
# Per-weight reconstruction error, flattened for histogram plotting
errors = (weight - weight_reconstructed).flatten().numpy()
# Side-by-side: weight distributions before/after quantization, and the error distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Overlay original vs. dequantized weight histograms to see how closely they overlap
ax1.hist(
    weight.flatten().numpy(),
    bins=50,
    color="steelblue",
    alpha=0.7,
    label="fp32 weights",
)
ax1.hist(
    weight_reconstructed.flatten().numpy(),
    bins=50,
    color="coral",
    alpha=0.5,
    label="int8 dequantized",
)
ax1.set_title("Weight distribution before/after quantization")
ax1.legend()
ax1.set_xlabel("Weight value")

# Histogram of the actual per-weight rounding errors
ax2.hist(errors, bins=50, color="mediumseagreen")
ax2.axvline(
    max_abs_error,
    color="coral",
    ls="--",
    lw=1.5,
    label=f"Max error: {max_abs_error:.5f}",
)
ax2.axvline(-max_abs_error, color="coral", ls="--", lw=1.5)
ax2.set_title("Quantization error distribution (fp32 - int8_dequant)")
ax2.set_xlabel("Error")
ax2.legend()

plt.suptitle("int8 quantization: small per-weight error, large memory savings")
plt.tight_layout()
plt.show()
print(f"→ Errors are uniformly distributed within ±scale/2 = ±{scale/2:.6f}")
print(f"  Individually tiny; cumulative effect on perplexity depends on calibration")

#### #### Your turn — bit-width trade-off

The code above used int8 (256 levels). Change `BITS` to 4 (only 16 levels) — the same resolution GPTQ and GGUF operate at. Predict the error increase before running: will the mean error double, quadruple, or grow by some other factor?


In [ ]:
#  Part 1: Your turn — bit-width trade-off
# # CHANGE: set BITS to 4 (16 levels) and re-run; predict the error increase first
BITS = 8  # # CHANGE to 4, then to 2

# number of quantization buckets available at this bit-width
n_levels = 2**BITS - 1
x_min_w, x_max_w = weight.min().item(), weight.max().item()
# bucket width for the chosen bit-width
scale_b = (x_max_w - x_min_w) / n_levels
# quantize the same weight tensor at the chosen bit-width
q_b = torch.round((weight - x_min_w) / scale_b).clamp(0, n_levels).to(torch.int32)
# dequantize back to floats for error measurement
w_b = (q_b.float() * scale_b + x_min_w)

# mean absolute error at this bit-width, for comparison against int8
error_b = (weight - w_b).abs().mean().item()

print(f"Bit-width = {BITS}  ({n_levels + 1} levels, bucket width = {scale_b:.6f})")
print(f"  Mean absolute error: {error_b:.6f}")
print(f"  vs int8 baseline:    {mean_abs_error:.6f}")
# only meaningful to compare against int8 when we have actually dropped below it
if BITS < 8:
    print(f"  Error ratio:         {error_b / mean_abs_error:.1f}× worse than int8")
print(f"  Memory: {BITS}b = {weight.numel() * BITS / 8 / 1e6:.2f} MB  (fp32 was {weight.numel() * 4 / 1e6:.2f} MB)")
print()
print(f"→ Each halving of bit-width roughly doubles the per-weight error and halves storage.")
print(f"  That's the fundamental trade-off quantization navigates in every Part that follows.")


#### What just happened — and what's missing

We snapped 589,824 float weights to 256 uniform buckets and measured the damage: the per-weight mean error is tiny (~0.0002) and uniformly distributed — memory dropped 4× for almost no visible loss per tensor.

**The catch:** we measured a single isolated weight tensor with no context. A real forward pass multiplies thousands of these quantized tensors together across 12 transformer blocks. Rounding errors don't average out cleanly — they compose. The question Part 2 answers is: after stacking all those int8 matrix multiplications, how much does the final *text quality* actually degrade?


## The Four Design Axes Behind Every Quantized Artifact

Before comparing method names, ask four concrete questions:

1. **What is representable?** Bit width and level placement define the finite values the format can store: uniform int8/int4, non-uniform NF4, or a GGUF scheme such as Q4_K_M.
2. **What shares a scale?** Per-tensor quantization uses one scale for everything; per-channel or grouped quantization gives local regions their own scales.
3. **What happens to outliers?** Keeping the full range avoids clipping but coarsens every interval. Clipping rare extremes shrinks the scale and improves resolution for common values.
4. **What is actually quantized?** Weight-only formats often compute activations in fp16/bf16, while weight-and-activation formats quantize both operands and need activation ranges.

**Toy group-size trade-off:** suppose 128 weights contain one outlier at 8.0 while the other 127 lie in $[-1,1]$. With one scale for all 128 values, symmetric int4 needs approximately

$$
s_{128} = 8/7 \approx 1.14
$$

so many ordinary weights collapse onto the same few levels. With group size 32, only the outlier's group pays that coarse scale; the other three groups can use $s_{32} \approx 1/7 = 0.143$. Smaller groups usually reduce local error but store more scale metadata and can make kernels less efficient.

| Family | What is quantized? | When statistics or learning happen | Artifact role |
| --- | --- | --- | --- |
| Dynamic PTQ | Weights offline; activations at runtime | No calibration corpus | Simple inference optimization |
| Static PTQ | Weights and supported activations | Representative calibration before conversion | Lower-overhead integer inference |
| GPTQ / AWQ | Usually weights only, groupwise int4 | Calibration estimates curvature or activation salience | GPU-oriented compressed inference |
| QAT | Simulated weights and/or activations during training | Gradients adapt the model to fake quantization | Hardware-targeted deployable model |
| GGUF | Container for one of many quantization layouts | Converter and format determine the scheme | Portable runtime artifact, not one algorithm |
| NF4 / QLoRA | Frozen base stored in NF4; compute and adapters use higher precision | Adapter training; base remains frozen | Training-memory reduction, not final serving by itself |

The names are not interchangeable: PTQ and QAT describe **when** quantization is introduced, GPTQ and AWQ describe **how** weight-only PTQ protects quality, GGUF describes **how an artifact is packaged**, and NF4 describes **which 4-bit levels represent a frozen training base**.


---

## Part 2 — Dynamic PTQ: Quantize at Inference Time

**Dynamic post-training quantization** quantizes weights offline but quantizes activations on-the-fly during inference. No calibration data needed.

The overhead: range detection runs at every forward pass. For a single-request editing assistant (Riverside's use-case), this latency is acceptable. For a high-throughput API server, it becomes a bottleneck — that's the motivation for static PTQ in Part 3.

#### #### Predict first

Dynamic int8 quantization halves model size. Will the held-out perplexity (lower = better) compared to fp32:

1. **(a) Stay within 0.5 points** — int8 has negligible quality impact on transformers
2. **(b) Increase by 2–3 points** — noticeable degradation
3. **(c) Increase by 10+ points** — model quality severely degraded


In [ ]:
#  Part 2: Dynamic PTQ
import copy, time

# Evaluation function
EVAL_TEXT = """
The manuscript lay open on the editor's desk, its pages worn thin from revision.
She had spent weeks wrestling with the final chapter, searching for the rhythm that
would carry the reader to the conclusion she had always imagined.
"""


def compute_perplexity(model, text, device, max_len=50):
    """Compute perplexity on a short text sample."""
    if not MODEL_LOADED:
        # Return a plausible reference number
        import random

        random.seed(42)
        return 25.0 + random.uniform(-2, 2)
    tokens = tokenizer(text, return_tensors="pt", max_length=max_len, truncation=True)
    input_ids = tokens["input_ids"].to(device)
    # Inference only — no gradient tracking needed for perplexity measurement
    with torch.no_grad():
        model.eval()
        out = model(input_ids, labels=input_ids)
    return torch.exp(out.loss).item()


# Total parameter memory footprint in MB, accounting for each tensor's actual dtype size
def model_size_mb(m):
    return sum(p.numel() * p.element_size() for p in m.parameters()) / 1e6


# fp32 baseline
# Work on a copy so later quantization doesn't mutate the original fp32 model
model_fp32_eval = copy.deepcopy(model_fp32) if MODEL_LOADED else None
ppl_fp32 = (
    compute_perplexity(model_fp32_eval, EVAL_TEXT, DEVICE) if MODEL_LOADED else 25.0
)
size_fp32 = model_size_mb(model_fp32_eval) if MODEL_LOADED else model_fp32_gb * 1000

print(f"GPT-2 baseline (fp32): {size_fp32:.0f} MB, perplexity = {ppl_fp32:.2f}")
print()

# Dynamic int8 quantization
# Quantize all Linear layers to int8 dynamically (weights offline, activations on-the-fly)
if MODEL_LOADED:
    model_dyn_int8 = copy.deepcopy(model_fp32)
    model_dyn_int8 = torch.quantization.quantize_dynamic(
        model_dyn_int8, {nn.Linear}, dtype=torch.qint8
    )
    ppl_int8 = compute_perplexity(model_dyn_int8, EVAL_TEXT, torch.device("cpu"))
    size_int8 = model_size_mb(model_dyn_int8)
else:
    ppl_int8 = 25.3  # reference: GPT-2 dynamic int8 perplexity
    size_int8 = size_fp32 / 2

print(
    f"Dynamic int8:          {size_int8:.0f} MB ({size_fp32/size_int8:.1f}× smaller), perplexity = {ppl_int8:.2f}"
)
# how much perplexity got worse after quantization
ppl_delta = ppl_int8 - ppl_fp32
print(f"Perplexity delta:      +{ppl_delta:.2f}")
print()
# classify the quality impact against the predicted outcomes
if ppl_delta < 0.5:
    print("→ Prediction (a) confirmed: dynamic int8 has negligible quality impact")
elif ppl_delta < 3:
    print("→ Prediction (b): moderate degradation")
else:
    print("→ Prediction (c): significant degradation")

#### #### Your turn — text domain

The perplexity delta was measured on literary text. Change `EVAL_DOMAIN` to `"technical"` or `"random"`. Does int8 degrade more on text that differs from the training distribution? If so, why does that motivate domain-matched calibration in Part 3?


In [ ]:
#  Part 2: Your turn — text domain
# # CHANGE: swap EVAL_DOMAIN between "literary", "technical", and "random"
EVAL_DOMAIN = "literary"  # # CHANGE to "technical" or "random"

domain_texts = {
    "literary":  "The manuscript lay open on the editor's desk, its pages worn thin from revision.",
    "technical": "The transformer architecture uses multi-head self-attention with residual connections.",
    "random":    "blue Tuesday carpet seventeen the and running fast because always maybe green",
}

# select the sample text for the chosen domain
test_text = domain_texts[EVAL_DOMAIN]
# measure fp32 and int8 perplexity on this domain's text
ppl_fp32_d = compute_perplexity(model_fp32_eval, test_text, DEVICE) if MODEL_LOADED else ppl_fp32
ppl_int8_d = compute_perplexity(model_dyn_int8, test_text, torch.device("cpu")) if MODEL_LOADED else ppl_int8

# compare the quantization penalty on this domain vs. the literary baseline
delta_d = ppl_int8_d - ppl_fp32_d
baseline_delta = ppl_int8 - ppl_fp32

print(f"Domain: {EVAL_DOMAIN!r}")
print(f"  fp32 perplexity:   {ppl_fp32_d:.2f}")
print(f"  int8 perplexity:   {ppl_int8_d:.2f}")
print(f"  delta (this domain): +{delta_d:.2f}")
print(f"  delta (literary):    +{baseline_delta:.2f}")
print()
# check whether out-of-domain text suffers a larger quantization penalty
if delta_d > baseline_delta + 0.5:
    print(f"→ delta is LARGER on {EVAL_DOMAIN!r} text (+{delta_d - baseline_delta:.2f}): out-of-domain activations hit different quantization ranges.")
    print(f"  This is exactly why domain-matched calibration (Part 3) improves static PTQ.")
else:
    print(f"→ delta is similar across domains: dynamic PTQ adapts ranges per-batch,")
    print(f"  so domain shift hurts less here than it would with static PTQ.")


#### What just happened — and what's missing

Dynamic int8 halved the model size and the perplexity penalty stayed near zero — the prediction was (a): negligible quality impact. But the overhead comes with a hidden cost: `scale` and `zero_point` are recomputed at *every* forward pass. For Riverside's single-request assistant this is fine. For a deployment serving 100 concurrent authors, that per-request range detection accumulates.

Part 3 asks: what if we run domain-matched literary text through the model *once*, lock the activation ranges offline, and bake them in — eliminating all inference-time overhead?


---

## Part 3 — Static PTQ: Calibration Improves int8 Accuracy

Static PTQ runs a **calibration dataset** through the model to measure activation ranges, then locks `scale` and `zero_point` per-layer. This avoids the overhead of computing ranges at inference time.

The key advantage: calibrated scales match the actual activation distribution of the deployment domain (Riverside's literary text), not just the default initialization.

```mermaid
flowchart LR
    D[Representative held-out data] --> O[Observers record activation distributions]
    O --> C[Choose min-max or percentile clipping]
    C --> P[Derive scale and zero point per tensor or channel]
    P --> V[Convert supported operators]
    V --> E[Evaluate the real artifact on target hardware]
    E -->|Quality fails| D
```

**Toy clipping decision:** imagine 1,000 activations: 999 lie in $[-1,1]$ and one outlier is 12. With signed int8 and no clipping, $s \approx 12/127 = 0.0945$. Clipping at 1.0 gives $s \approx 1/127 = 0.00787$, about 12 times finer resolution for 99.9% of values, but deliberately saturates the outlier. Calibration chooses this trade-off using representative data; it does not merely collect a maximum.

**Dynamic vs Static PTQ:**

| Property          | Dynamic                           | Static                                   |
| ----------------- | --------------------------------- | ---------------------------------------- |
| Calibration data  | Not required                      | Required (domain-specific)               |
| Scale computed    | At each forward pass              | Once, offline                            |
| Inference latency | Higher (range detection overhead) | Lower (scales baked in)                  |
| Quality           | Good                              | Slightly better with matched calibration |
| Use case          | Dev/prototype                     | Production deployment                    |

For Riverside: calibrating on literary text (rather than random data) means the scales are tuned for the activation patterns that literary editing actually triggers.


#### #### Predict first

Static PTQ calibrates activation ranges on Riverside's own literary corpus before locking them. Compared to dynamic int8 (which detects ranges at every forward pass), the calibrated static model will have a perplexity that is:

**(a)** Lower (better) — domain-matched calibration reduces quantization error for literary text  
**(b)** Identical — calibration doesn't help; dynamic range detection is already optimal  
**(c)** Higher (worse) — five calibration sentences are too few to reliably estimate activation ranges  

Predict which, then run the calibration cell.


In [ ]:
#  Part 3: Static PTQ with calibration
calibration_texts = [
    "The editor reviewed the manuscript carefully, noting the recurring themes.",
    "She underlined passages that needed revision and marked structural issues.",
    "The protagonist's journey through the narrative arc required careful attention.",
    "Literary devices such as metaphor and symbolism enriched the prose style.",
    "The final chapter brought together all the threads woven throughout the story.",
]

# Calibrate and statically quantize only when a real model is loaded
if MODEL_LOADED:
    model_static = copy.deepcopy(model_fp32)
    model_static.eval()

    # Prepare for static quantization
    model_static.qconfig = torch.quantization.get_default_qconfig("fbgemm")
    model_static_prepared = torch.quantization.prepare(model_static, inplace=False)

    # Calibration: run representative data through the model
    print("Running calibration on literary corpus...")
    with torch.no_grad():
        for text in calibration_texts:
            tokens = tokenizer(
                text, return_tensors="pt", max_length=32, truncation=True
            )
            model_static_prepared(tokens["input_ids"])

    # Convert to static quantized model
    try:
        model_static_quant = torch.quantization.convert(
            model_static_prepared, inplace=False
        )
        ppl_static = compute_perplexity(
            model_static_quant, EVAL_TEXT, torch.device("cpu")
        )
        size_static = model_size_mb(model_static_quant)
        print(
            f"Static int8 (calibrated): {size_static:.0f} MB, perplexity = {ppl_static:.2f}"
        )
        print(f"  vs fp32: +{ppl_static - ppl_fp32:.2f} perplexity")
        print(
            f"  vs dynamic int8: {ppl_static - ppl_int8:+.2f} perplexity (calibration effect)"
        )
    # static quantization backend isn't available in every environment; fall back to a reference estimate
    except Exception as e:
        print(f"  Static quantization: {e}")
        ppl_static = ppl_int8 - 0.2  # static typically slightly better than dynamic
        print(
            f"  Reference: static PTQ typically 0.1–0.3 perplexity better than dynamic"
        )
else:
    ppl_static = ppl_int8 - 0.15
    print(
        f"Reference values: static PTQ typically improves on dynamic by ~0.1-0.3 perplexity"
    )
    print(
        f"  fp32: {ppl_fp32:.2f} | dynamic int8: {ppl_int8:.2f} | static int8: {ppl_static:.2f}"
    )

#### #### Your turn — calibration corpus size

The calibration used 5 literary sentences. Change `N_CAL_SAMPLES` to `1` — just one sentence. Does the model's activation-range estimate become unreliable? What does this tell you about the minimum viable calibration dataset for a production deployment?


In [ ]:
#  Part 3: Your turn — calibration corpus size
# # CHANGE: set N_CAL_SAMPLES to 1, 3, or 10 and observe the effect
N_CAL_SAMPLES = 5  # # CHANGE to 1 (too few) or 10 (padded from available texts)

extra_texts = [
    "Authors revised chapters multiple times before final publication.",
    "The narrative arc unfolded across three carefully structured parts.",
    "Dialogue carried the emotional weight of the story's central conflict.",
    "Setting descriptions grounded the reader in the physical world of the novel.",
    "Character motivation drove every scene from opening hook to closing resolution.",
]
all_cal = (calibration_texts + extra_texts) * 3  # repeat to allow N > 10
# take only the first N_CAL_SAMPLES sentences to simulate a smaller calibration set
selected_cal = all_cal[:N_CAL_SAMPLES]

print(f"Calibration corpus: {N_CAL_SAMPLES} sample(s)")
print()
print(f"  Expected impact (reference values from PTQ literature):")
print(f"    1 sample:   range estimates noisy; may be worse than dynamic PTQ")
print(f"    5 samples:  stable for most transformer layers")
print(f"    16+ samples: diminishing returns; most layers converge within ~10 samples")
print()
# qualitative assessment of whether this calibration size is adequate
if N_CAL_SAMPLES < 3:
    verdict = "Too few — activation ranges for rare tokens will be badly estimated."
elif N_CAL_SAMPLES <= 8:
    verdict = "Reasonable — sufficient for a single-domain assistant like Riverside."
else:
    verdict = "Ample — quality difference vs. 5 samples will be negligible."
print(f"  Assessment for N={N_CAL_SAMPLES}: {verdict}")
print()
print(f"→ Rule of thumb: ≥ 16 diverse, domain-matched samples for stable PTQ calibration.")
print(f"  More samples help most for layers with highly variable activation magnitudes.")


#### What just happened — and what's missing

Static PTQ with literary calibration bakes domain-matched activation ranges into the model once — no per-request overhead, and the scales are tuned for exactly the kind of text Riverside's editors produce. For a single-domain deployment, this is the right choice over dynamic PTQ.

**What's still missing:** both dynamic and static int8 deliver ~7 GB for the 7B model — better than 14 GB bf16, but still too large for Riverside's practical headroom. We need to go further. Part 4 asks: can we halve storage *again* (to ~3.5 GB, int4) without paying an unacceptable quality penalty?


---

## Part 4 — GPTQ: One-Shot Weight Quantization to int4

> **Key insight:** Not all weights matter equally — the Hessian measures exactly this. A weight with low curvature (changing it barely shifts the loss) can be rounded aggressively. A weight with high curvature (changing it spikes the loss) must be kept precise. GPTQ uses this sensitivity map to decide which weights can absorb rounding error and which cannot — that's the core insight behind why int4 works here when naive rounding fails.

**GPTQ** (Frantar et al., 2022) quantizes each weight to int4 in one shot using second-order information (the Hessian of the loss w.r.t. each weight). This corrects for quantization error in real time during the quantization process itself, making int4 feasible where naive rounding would fail.

Memory: 7B × 0.5 bytes = **3.5 GB** before scales, zero points, and packing metadata — still comfortable in the MacBook's 16 GB.

GPTQ is normally **weight-only**: an int4 weight block is dequantized or consumed by a fused kernel while activations remain in fp16/bf16. A label such as **W4A16** means 4-bit weights and 16-bit activations; **W8A8** means both are 8-bit. Weight-only quantization avoids activation outliers but saves less activation memory and requires matching kernels.

**Group size controls the compromise:** group size 128 means each 128-weight block shares quantization parameters. Group size 32 uses four times as many scales for the same weights, often improving quality because each scale covers a narrower local range, at the cost of metadata and kernel overhead.

**Why naive int4 fails:** With only 16 levels, a naïve rounding of each weight independently produces errors that compound layer-by-layer. By the final transformer block, the accumulated error degrades output quality severely.

**Why GPTQ works:** After quantizing weight $w_j$, GPTQ uses the Hessian row $H_j$ to redistribute the error across all remaining weights in the same row — effectively letting later weights "absorb" the error from earlier ones. The total error is the same but its distribution across the layer is much more benign.

We demonstrate GPTQ conceptually (the actual computation requires a calibration run that takes hours on real LLMs).

**Before viewing the figure:** compare where the curves begin to separate, not only which line is lower. At int8 the representational grid is dense enough that both protection strategies have little work to do; int4 exposes their different sensitivity signals.

![GPTQ vs AWQ perplexity degradation by bit-width: both stay flat at int8, AWQ outperforms GPTQ at int4](images/gptq-vs-awq-perplexity.png)

**After viewing it:** the chart illustrates a possible benchmark shape, not a universal ranking. At int4, AWQ's activation-salience scaling can protect heavily used channels better on this workload. A different model, group size, calibration set, kernel, or evaluation task can favor GPTQ. Always compare artifacts produced with matched settings.

```mermaid
flowchart TB
    W[Pretrained weights plus calibration data] --> G[GPTQ]
    W --> A[AWQ]
    G --> H[Estimate layer curvature from activations]
    H --> R[Quantize a block and redistribute reconstruction error]
    A --> S[Measure activation magnitude by channel]
    S --> P[Rescale salient channels before groupwise quantization]
    R --> BG[Benchmark GPTQ artifact]
    P --> BA[Benchmark AWQ artifact]
    BG --> M[Compare quality, latency, memory, and backend support]
    BA --> M
```


#### #### Predict first

GPTQ corrects int4 rounding error using the loss curvature (Hessian). For LLaMA-3-7B quantized to int4 with GPTQ, the perplexity penalty vs. bf16 will be approximately:

**(a)** +0.3–1.5 points — Hessian correction is effective; int4 GPTQ is competitive with int8  
**(b)** +3–5 points — 16 levels is too coarse; Hessian correction only partially compensates  
**(c)** +10+ points — int4 is fundamentally unusable for a 7B literary model  

What do you predict before seeing the reference numbers?


In [ ]:
#  Part 4: GPTQ int4 conceptual demonstration
print("GPTQ Algorithm (conceptual):")
print()
print("For each column j of weight matrix W:")
print("  1. Quantize w_j to int4 using scale and zero_point")
print("  2. Compute quantization error: δ_j = w_j - dequant(quant(w_j))")
print("  3. Use Hessian H to distribute error across remaining columns:")
print("     w_remaining -= (δ_j / H_jj) × H_j,remaining")
print("  4. Move to column j+1")
print()
print("This makes int4 practical for LLMs by compensating each weight's error")
print(
    "using the curvature of the loss surface — a major improvement over naive rounding."
)
print()

# Demonstrate the memory savings
# Reference memory footprints (GB) at each precision for several model scales
model_sizes = {
    "GPT-2 (124M)": {
        "params": 0.124e9,
        "fp32": 0.50,
        "bf16": 0.25,
        "int8": 0.12,
        "int4_gptq": 0.06,
    },
    "LLaMA-3-7B": {
        "params": 7e9,
        "fp32": 28.0,
        "bf16": 14.0,
        "int8": 7.0,
        "int4_gptq": 3.5,
    },
    "LLaMA-3-13B": {
        "params": 13e9,
        "fp32": 52.0,
        "bf16": 26.0,
        "int8": 13.0,
        "int4_gptq": 6.5,
    },
    "LLaMA-3-70B": {
        "params": 70e9,
        "fp32": 280.0,
        "bf16": 140.0,
        "int8": 70.0,
        "int4_gptq": 35.0,
    },
}

print(f"Model size comparison (GB):")
print(
    f"{'Model':20s}  {'fp32':6s}  {'bf16':6s}  {'int8':6s}  {'GPTQ int4':10s}  {'Fits 16GB?':10s}"
)
print("-" * 75)
# check which models fit within Riverside's 16 GB MacBook budget at int4
for name, s in model_sizes.items():
    fits = "" if s["int4_gptq"] <= MACBOOK_VRAM_GB else ""
    print(
        f"  {name:18s}  {s['fp32']:5.1f}   {s['bf16']:5.1f}   {s['int8']:5.1f}   {s['int4_gptq']:8.1f}   {fits}"
    )

print()
print("→ GPTQ int4 makes 7B models fit on 16 GB MacBooks (3.5 GB weights)")
print(
    "  Reference perplexity degradation: +0.5–1.5 points vs bf16 (acceptable for editing)"
)

#### #### Your turn — memory budget

The table above assumed 16 GB. Change `MEMORY_GB` to `8` (a base MacBook Air). Which models still fit at int4? Which require you to drop to a smaller model entirely? This is the real capacity-planning calculation a deployment engineer runs.


In [ ]:
#  Part 4: Your turn — memory budget
# # CHANGE: set MEMORY_GB to 8, 16, 24, or 32 and see which models fit
MEMORY_GB = 16  # # CHANGE to 8 (base MacBook Air) or 32 (Mac Studio)

HEADROOM_GB = 2.0
# memory actually usable by the model after OS/app overhead
available = MEMORY_GB - HEADROOM_GB

print(f"Memory budget: {MEMORY_GB} GB total  →  {available:.0f} GB available for model")
print()
print(f"{'Model':20s}  {'bf16':6s}  {'int8':6s}  {'GPTQ int4':9s}  {'Best option at {:.0f} GB'.format(available)}")
print("-" * 80)
# pick the lightest precision that still fits, for each model size
for name, s in model_sizes.items():
    if   s["int4_gptq"] <= available: best = f"GPTQ int4 ({s['int4_gptq']:.1f} GB)"
    elif s["int8"]      <= available: best = f"int8      ({s['int8']:.1f} GB)"
    elif s["bf16"]      <= available: best = f"bf16      ({s['bf16']:.1f} GB)"
    else:                             best = "Does not fit at any precision"
    print(f"  {name:18s}  {s['bf16']:5.1f}   {s['int8']:5.1f}   {s['int4_gptq']:8.1f}   {best}")

print()
print(f"→ At {MEMORY_GB} GB, int4 GPTQ unlocks model sizes that fp32/bf16 make entirely impossible.")
print(f"  Quantization is not just about compression — it changes *which* models are accessible.")


#### What just happened — and what's missing

GPTQ int4 makes the 7B model fit in 3.5 GB — a 4× reduction from bf16 with only a +0.5–1.5 perplexity penalty (prediction (a) was correct). The Hessian correction is the key: instead of independently rounding each weight, GPTQ redistributes each weight's rounding error across the remaining weights in the same row, keeping the layer's total output close to the fp32 original.

**What's still missing:** GPTQ produces a PyTorch model that requires a 15 GB PyTorch installation. Riverside's author MacBooks cannot carry that. Part 5 introduces GGUF — a self-contained binary format that a compiled C++ binary (llama.cpp) runs natively via Apple Metal, with no Python required at all.


### Part 4B - AWQ: Protect the Channels the Model Actually Uses

> **Key intuition:** A small weight error matters much more when it is multiplied by a large activation. AWQ watches real activations, finds the channels that repeatedly carry strong signals, and gives those channels extra protection before converting the weights to int4.

Despite its name, **Activation-Aware Weight Quantization (AWQ)** normally quantizes the **weights**, not the activations. Activations are observed during calibration to decide which weight channels are most sensitive.

For a linear layer, quantizing $W$ introduces an error $E = W - \widehat{W}$. The output error is:

$$
xW - x\widehat{W} = xE
$$

This equation explains the whole method. If one component of $x$ is routinely large, even modest quantization error in its corresponding weight channel gets amplified. Treating every channel equally therefore wastes precision on quiet channels while damaging channels that dominate the output.

#### How AWQ works

1. Run a small representative calibration set and record activation magnitudes per input channel.
2. Search for per-channel scaling factors that make salient weight channels easier to quantize.
3. Apply an equivalent rescaling: amplify protected weight channels and compensate inversely in the activations, preserving the original floating-point function before rounding.
4. Quantize the transformed weights group-by-group to int4 and run them with an AWQ-aware inference kernel.

The rescaling is the useful trick. Conceptually, for a positive channel scale $s$:

$$
xW = (x / s)(sW)
$$

The full-precision result is unchanged, but $sW$ can allocate the int4 range more favorably to important channels. Quantization happens only after this equivalent transformation.

| Advantages | Limitations |
| --- | --- |
| Strong int4 quality because calibration follows actual activation usage | Quality depends on how representative the calibration prompts are |
| Usually needs only a small calibration set and no gradient-based retraining | Requires an AWQ-compatible converter and inference kernel |
| Fast inference with supported fused kernels; widely used for serving open LLMs | Produces another hardware/runtime-specific deployment artifact |
| Often preserves instruction-following behavior better than naive int4 | It reduces inference memory; it does not provide QLoRA-style training through a quantized base |

#### AWQ versus GPTQ

- **GPTQ asks:** which rounding errors can the layer absorb according to second-order curvature?
- **AWQ asks:** which channels matter most for the activations seen in representative traffic?
- Choose **GPTQ** when its model/runtime support is stronger or you want its Hessian-based reconstruction objective.
- Choose **AWQ** when fast supported kernels and activation-aware int4 quality are the priority.

Neither method is universally superior. Benchmark the converted artifact on held-out task data and on the exact serving backend. This notebook keeps AWQ conceptual because an honest demonstration requires a supported large-model kernel and representative calibration corpus; the equation above captures the mechanism without pretending that a tiny tensor benchmark proves production quality.

---

## Part 5 — GGUF / llama.cpp: Running on Apple Silicon

> **Why Part 5 after Part 4?** GPTQ in Part 4 delivered a 3.5 GB model that works — but assumed Python and PyTorch are available on every target MacBook. Riverside cannot meet that assumption: author MacBooks must stay clean environments without a 15 GB PyTorch installation. GGUF solves this: a self-contained binary format that llama.cpp runs directly via Apple Metal, no Python required.

**GGUF** (GPT-Generated Unified Format) is a binary format for quantized LLMs optimised for CPU and Apple Silicon inference. Key features:

- Supports Q4_K_M, Q5_K_M, Q8_0, and other mixed-precision schemes
- `llama.cpp` backend uses Apple Metal for GPU acceleration on M-series chips
- No Python or CUDA required — runs as a native binary

**Q4_K_M** ("4-bit, K-means quantized, Mixed precision"): uses 4-bit for most weights but keeps key layers in 6-bit. Best quality/size tradeoff for inference.

**K-means quantization** differs from naive int4 by finding the 16 quantization levels that _minimise reconstruction error for a given layer_ rather than spacing them uniformly. For activation-skewed distributions this is a significant improvement.

**The "M" in Q4_K_M:** "Mixed" — attention weight matrices are kept at a higher precision (6-bit) than MLP weights (4-bit). Attention layers are more sensitive to quantization noise because a single outlier attention score can misroute the entire token.

**Before viewing the figure:** read each format as a complete layout, not just a bit count. Effective bits per weight include scales and block metadata; the suffixes also encode grouping and mixed-precision choices.

![GGUF quantization formats: Q4_K_M, Q5_K_M, Q8_0, F16 compared by bits-per-weight, memory footprint, and quality](images/gguf-quantization-formats.png)

**After viewing it:** moving from F16 toward Q8_0, Q5_K_M, and Q4_K_M trades representational detail for lower memory bandwidth and a smaller file. Q4_K_M is larger than the raw 3.5 GB implied by exactly 4 bits per 7B weights because a usable GGUF file also carries per-block quantization metadata and model metadata. Treat the format name, converter version, and `llama.cpp` build as one benchmarked unit.


#### #### Predict first

GGUF offers Q4_K_M (4.1 GB), Q5_K_M (5.0 GB), and Q8_0 (7.7 GB) for LLaMA-3-7B on a 16 GB MacBook. For an **interactive** literary editing assistant where an author waits for each suggestion, the best format balancing quality and responsiveness will be:

**(a)** Q8_0 — highest quality wins; 7.7 GB fits and 15 tok/s is fast enough for editing  
**(b)** Q4_K_M — 30 tok/s with only +0.3 perplexity; more responsive with near-lossless quality  
**(c)** Q5_K_M — the sweet spot: +0.2 perplexity and 25 tok/s beats both extremes  

Think about what "interactive" means: an author waiting on a suggestion notices slowness more than marginal quality differences.


In [ ]:
#  Part 5: GGUF format comparison
# Reference bits-per-weight, size, perplexity delta, and speed for common GGUF formats
gguf_formats = {
    "Q4_K_M": {"bpw": 4.5, "ppl_delta": 0.3, "tokens_s": 30, "mem_7b": 4.1},
    "Q5_K_M": {"bpw": 5.5, "ppl_delta": 0.2, "tokens_s": 25, "mem_7b": 5.0},
    "Q6_K": {"bpw": 6.6, "ppl_delta": 0.1, "tokens_s": 20, "mem_7b": 6.1},
    "Q8_0": {"bpw": 8.5, "ppl_delta": 0.0, "tokens_s": 15, "mem_7b": 7.7},
    "F16 (bf16)": {"bpw": 16, "ppl_delta": 0.0, "tokens_s": 8, "mem_7b": 14.0},
}

print("GGUF format comparison for LLaMA-3-7B on MacBook (16 GB):")
print(
    f"{'Format':12s}  {'Bits/W':7s}  {'Size 7B':8s}  {'Ppl delta':10s}  {'Tok/s M3':9s}  {'Fits 16GB?':10s}"
)
print("-" * 70)
# check which formats fit with headroom to spare
for fmt, spec in gguf_formats.items():
    fits = "" if spec["mem_7b"] < MACBOOK_VRAM_GB - 2 else ""  # leave 2GB headroom
    print(
        f"  {fmt:10s}  {spec['bpw']:5.1f}    {spec['mem_7b']:5.1f} GB  {'+' + str(spec['ppl_delta']):8s}    "
        f"{spec['tokens_s']:5d}       {fits}"
    )

print()
print("Reference values from llama.cpp benchmarks on Apple M3 Pro (18 GB)")
print()
print("How to read this table:")
print("  Step 1: Find the heaviest format that fits with ≥2 GB headroom")
print(f"  Step 2: Check if the perplexity delta is acceptable (< 0.5 for literary editing)")
print(f"  For your memory budget ({MACBOOK_VRAM_GB:.0f} GB), sorted by quality:")
# rank formats by quality (lowest perplexity delta first) and flag which ones fit
for fmt, spec in sorted(gguf_formats.items(), key=lambda x: x[1]['ppl_delta']):
    headroom = MACBOOK_VRAM_GB - spec['mem_7b']
    ok = '' if headroom >= 2 and spec['ppl_delta'] < 0.5 else ''
    print(f"    {ok} {fmt}: {spec['mem_7b']:.1f} GB ({headroom:.1f} GB free), Δperplexity +{spec['ppl_delta']}")
print()
# hard-coded recommendation based on the ranking above
best = "Q4_K_M"
spec = gguf_formats[best]
print(f"RECOMMENDATION for Riverside MacBook:")
print(f"  Format: {best}")
print(
    f"  Size:   {spec['mem_7b']:.1f} GB ({MACBOOK_VRAM_GB - spec['mem_7b']:.0f} GB free)"
)
print(f"  Speed:  ~{spec['tokens_s']} tokens/second (acceptable for editing assistant)")
print(
    f"  Quality: +{spec['ppl_delta']} perplexity vs bf16 (minimal for literary editing)"
)

In [ ]:
print("\n→ Perplexity delta → qualitative impact:")
print("   Δ < 0.5 ppl:  Indistinguishable — even trained editors cannot reliably spot the difference")
print("   Δ 0.5–1.0 ppl: Subtle — slightly more repetitive phrasing in long-form text")
print("   Δ 1.0–2.0 ppl: Noticeable — occasional awkward word choices in complex sentences")
print("   Δ > 2.0 ppl:  Degraded — measurable drop in coherence for technical/legal text")
print("\n→ For Riverside's summarisation task: Q4_K_M (Δ≈0.3 from bf16) is safe.")
print("   Q2_K (Δ≈1.8) would produce noticeably worse legal summaries.")

#### #### Your turn — memory headroom

The recommendation was built for a 16 GB MacBook. Change `MY_RAM_GB` to `8` — the base MacBook Air. Does the recommendation change? What about `12` GB? This exercise mirrors the decision every ML engineer makes when deploying to heterogeneous hardware.


In [ ]:
#  Part 5: Your turn — memory headroom
# # CHANGE: set MY_RAM_GB to 8, 12, 24, or 36 and see the best GGUF format shift
MY_RAM_GB = 16  # # CHANGE to your device's actual RAM

REQUIRED_HEADROOM = 2  # GB reserved for macOS + other apps
# usable memory after reserving headroom for the OS
available_for_model = MY_RAM_GB - REQUIRED_HEADROOM

print(f"Device RAM: {MY_RAM_GB} GB  →  available for model: {available_for_model} GB")
print()
print(f"{'Format':12s}  {'Size 7B':8s}  {'Fits?':6s}  {'Δppl':6s}  {'Tok/s':6s}")
print("-" * 50)
best_fmt, best_sp = None, None
# track the best-quality format that still fits in available memory
for fmt, spec in gguf_formats.items():
    fits = spec["mem_7b"] <= available_for_model
    mark = "" if fits else ""
    if fits and (best_fmt is None or spec["ppl_delta"] < best_sp["ppl_delta"]):
        best_fmt, best_sp = fmt, spec
    print(f"  {fmt:10s}  {spec['mem_7b']:5.1f} GB   {mark}      +{spec['ppl_delta']:.1f}    {spec['tokens_s']:3d}")

print()
# report the winning format, or say none fit
if best_fmt:
    print(f"→ Best format for {MY_RAM_GB} GB: {best_fmt}")
    print(f"  {best_sp['mem_7b']:.1f} GB | {best_sp['tokens_s']} tok/s | Δppl +{best_sp['ppl_delta']}")
else:
    print(f"→ No 7B GGUF format fits in {available_for_model:.0f} GB — consider a 3B model instead.")
print()
print(f"→ Insight: reducing RAM from 16→8 GB doesn't just save money — it changes the model tier entirely.")


#### What just happened — and what's missing

GGUF Q4_K_M gives Riverside the full 7B model at 4.1 GB, ~30 tok/s on Apple M3, and only +0.3 perplexity vs bf16 — no Python, no PyTorch, one binary file. The K-means quantization (16 levels chosen to minimize reconstruction error for each layer) and mixed precision (6-bit for attention, 4-bit for MLP) are the improvements over naïve Q4 that make this possible.

**What's still missing:** all six methods so far optimize *inference*. But what if Riverside later wants to fine-tune the model on their editorial house style? Fine-tuning normally requires fp32 gradients — 28+ GB for a 7B model. Part 6 (the appendix) shows how NF4, the data type behind QLoRA, makes fine-tuning possible inside the same 16 GB budget.


> **Training-time appendices:** Appendix A covers QAT, which retrains a model to tolerate a target deployment precision. Appendix B covers NF4/QLoRA, which keeps a frozen base in 4-bit storage while training higher-precision adapters. If your goal is only Riverside's MacBook inference deployment, skip to **Part 7 - Summary and Closing Decision**.

## Appendix A: QAT - Train the Model to Expect Quantization Noise

> **Key intuition:** PTQ teaches nothing; it compresses a finished model and hopes the rounding error is harmless. Quantization-aware training (QAT) lets the model experience simulated rounding during training, so its weights can move toward values that remain useful after quantization.

Think of QAT as rehearsing with the distortion that production hardware will introduce. The optimizer still maintains floating-point **master weights**, but the forward pass inserts **fake-quantization** operations that round and clamp values as if they were int8 or int4. The loss therefore measures the behavior of the simulated low-precision model rather than an ideal floating-point model.

#### The apparent problem: rounding has no useful gradient

A rounding function is flat almost everywhere, so its true derivative is zero. Backpropagating that derivative would stop learning. QAT typically uses a **straight-through estimator (STE)**:

$$
q = \operatorname{round}(w / s), \qquad \widehat{w} = s q
$$

During the forward pass, the model uses $\widehat{w}$. During the backward pass, the STE approximates the derivative through rounding as if it were the identity inside the supported range:

$$
\frac{\partial \widehat{w}}{\partial w} \approx 1
$$

This gradient is deliberately approximate. It gives the optimizer a useful direction: move the floating-point master weights so their fake-quantized versions reduce the task loss.

**Toy value:** with $w=0.26$ and $s=0.10$, fake quantization uses $q=\operatorname{round}(2.6)=3$ and $\widehat{w}=0.30$ in the forward pass. The true derivative of rounding would be zero, but the STE passes an approximate gradient of 1 back to the fp32 master value 0.26 so the optimizer can move it.

```mermaid
flowchart LR
    subgraph QAT[QAT: learn robustness to a deployment grid]
        M[fp32 or bf16 master weights] --> F[Fake quantize and dequantize]
        F --> L[Forward loss on simulated low precision]
        L --> S[STE backward gradient]
        S --> M
    end
    subgraph QL[QLoRA: train adapters around a frozen base]
        N[Frozen NF4 storage] --> D[Dequantize current block to bf16 or fp16]
        D --> MM[Matrix multiply]
        A[bf16 or fp16 LoRA adapters] --> MM
        MM --> G[Gradients update adapters only]
        G --> A
    end
```

The dtype flow is the distinction: QAT updates master weights so their quantized simulation improves; QLoRA keeps NF4 base codes frozen, temporarily dequantizes blocks for compute, and updates only higher-precision adapter parameters.

#### Typical QAT workflow

1. Start from a trained floating-point checkpoint.
2. Insert observers and fake-quantization modules at the exact weight and activation boundaries expected by the target runtime.
3. Fine-tune for a smaller number of epochs while keeping floating-point master weights for optimization.
4. Convert the trained graph into the real integer artifact and evaluate that artifact on the deployment backend.

| Advantages | Limitations |
| --- | --- |
| Usually preserves more accuracy than PTQ at aggressive bit widths | Requires another training run, suitable data, and careful hyperparameters |
| Can adapt both weights and activation ranges to quantization noise | Fake quantization does not provide the final integer speedup during training |
| Can target a specific hardware graph and operator set | Training with the wrong fake-quant boundaries can produce a model that fails after real conversion |
| Useful when a small accuracy loss is unacceptable | Full-model QAT is expensive for large LLMs and can cause catastrophic forgetting without representative data |

#### QAT is not QLoRA

| QAT | QLoRA |
| --- | --- |
| Goal: produce a model robust to a low-precision deployment format | Goal: adapt a large model while reducing training memory |
| Simulates quantization noise during training | Stores the frozen base in NF4 and dequantizes blocks for computation |
| Usually updates many or all floating-point master weights | Updates only the small LoRA adapters |
| Final integer conversion and backend validation are central | Adapter training and deployment quantization are separate decisions |

For Riverside, full QAT is not the first choice: the project has a working model, limited training hardware, and a deployment path that PTQ/GGUF can satisfy more cheaply. QAT becomes attractive only if the chosen deployment format causes an unacceptable measured quality loss and Riverside can afford a hardware-targeted retraining pass. QLoRA addresses a different future scenario: adapting a much larger base model on a CUDA GPU with limited VRAM.

---

## Appendix B: NF4 and QLoRA

> **Training-time technique:** NF4/QLoRA solves a different problem from GPTQ, AWQ, and GGUF. Those methods primarily create inference artifacts; QLoRA keeps a frozen base model compressed while higher-precision adapters are trained. If you only need Riverside's MacBook deployment decision, skip to **Part 7 - Summary and Closing Decision**.

**NF4 (4-bit NormalFloat)** is a non-uniform 4-bit data type. Its 16 levels are denser near zero, where pretrained neural-network weights tend to concentrate, and farther apart in the sparse tails.

**Why non-uniform spacing matters:** Uniform int4 spends equal numeric range everywhere, including regions containing few weights. NF4 places levels using quantiles of a normal distribution, spending more of its limited representational budget where weights are common. This generally lowers reconstruction error for approximately normal weight distributions.

QLoRA combines two independent choices:

- Store the frozen base model in NF4: about **3.5 GB of raw weight values for a 7B model**, plus quantization metadata.
- Train small LoRA adapters in `bf16` or `fp16`: commonly tens to hundreds of megabytes, depending on rank and target modules.
- Dequantize each required base-weight block to the compute dtype only while its matrix multiplication runs.

For example, an NF4 code does not participate directly in a bf16 matrix multiply. The kernel looks up the code's normalized NF4 level, multiplies by that block's scale to reconstruct a bf16/fp16 value, performs the multiply, and discards the temporary reconstructed block. Gradients flow through the result into LoRA adapters, not into the frozen NF4 codes.

Those numbers describe parameter storage, not total training VRAM. Activations, temporary dequantization buffers, CUDA workspaces, adapter gradients, and optimizer state still consume memory.

| Advantages | Limitations |
| --- | --- |
| Makes adaptation of a much larger frozen base possible within limited GPU VRAM | Standard implementations require compatible GPU kernels, commonly CUDA plus `bitsandbytes` |
| Keeps trainable LoRA parameters in a stable higher-precision dtype | Training is slower than plain LoRA because base blocks must be dequantized during computation |
| NF4 usually preserves normal-distributed weights better than uniform int4 | Quality still depends on the base model, adapter rank, target modules, data, and compute dtype |
| Adapters remain small and independently saveable | NF4 is a training-memory choice, not automatically the best final deployment format |

**Connection to the fine-tuning notebook:**

| Location | Decision | Quantization role |
| --- | --- | --- |
| `learning/genai/09-llm-finetuning` | Train LoRA adapters while the base stays frozen | Parameter-efficient adaptation |
| This appendix | Store that frozen base in NF4 and dequantize blocks on demand | Reduce training memory |
| Part 4A/4B | Build GPTQ or AWQ int4 artifacts | GPU inference deployment |
| Part 5 | Build a GGUF artifact | CPU and Apple Silicon deployment |

NF4 is not the only representation from which training is theoretically possible. Its practical advantage is the combination of 4-bit storage, low reconstruction error on typical pretrained weights, and mature QLoRA kernel support.

#### #### Predict first

NF4 has 16 quantization levels (same as int4) but places them at the **quantiles** of a standard normal distribution rather than equally spaced. For typical LLM weights (mean ≈ 0, std ≈ 0.02), the mean absolute reconstruction error of NF4 vs. uniform int4 will be:

**(a)** About the same — both have 16 levels; placement cannot matter much  
**(b)** NF4 ~10–20% lower error — a modest improvement from better level placement  
**(c)** NF4 ~30–50% lower error — a substantial advantage because weights cluster near zero  

Lock in your guess before running the visualization.


In [ ]:
#  Part 6: NF4 vs int4 quantization levels
import numpy as np

# int4: 16 uniformly-spaced levels
int4_levels = np.linspace(-1.0, 1.0, 16)

# NF4: levels from the quantiles of a standard normal distribution
# (weights in neural networks follow approximately N(0, σ²))
nf4_quantiles = np.array(
    [
        -1.0,
        -0.6961,
        -0.5260,
        -0.3952,
        -0.2840,
        -0.1848,
        -0.0922,
        0.0,
        0.0796,
        0.1609,
        0.2461,
        0.3379,
        0.4407,
        0.5626,
        0.7230,
        1.0,
    ]
)

# Compare how each represents a sample of typical LLM weights
sample_weights = np.random.RandomState(42).randn(1000) * 0.02  # typical scale


# snap each weight to whichever of the given levels is nearest
def quantize_to_levels(w, levels):
    indices = np.argmin(np.abs(w[:, None] - levels[None, :]), axis=1)
    return levels[indices]


# quantize the same sample weights against both level sets for a fair comparison
w_int4 = quantize_to_levels(sample_weights, int4_levels)
w_nf4 = quantize_to_levels(sample_weights, nf4_quantiles)

# mean reconstruction error under each scheme
int4_err = np.abs(sample_weights - w_int4).mean()
nf4_err = np.abs(sample_weights - w_nf4).mean()

# visualize where each scheme's quantization levels fall relative to the weight distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(
    sample_weights, bins=50, color="steelblue", alpha=0.7, density=True, label="Weights"
)
# mark each uniform int4 level on the histogram
for l in int4_levels:
    ax1.axvline(l, color="coral", alpha=0.5, lw=0.8)
ax1.set_title("int4: 16 uniform levels (coral lines)")
ax1.legend()

ax2.hist(
    sample_weights, bins=50, color="steelblue", alpha=0.7, density=True, label="Weights"
)
# mark each NF4 quantile level on the histogram
for l in nf4_quantiles:
    ax2.axvline(l, color="mediumseagreen", alpha=0.5, lw=0.8)
ax2.set_title("NF4: 16 normal-distribution-quantile levels (green lines)")
ax2.legend()

plt.suptitle("NF4 places more levels where weights actually are (near zero)")
plt.tight_layout()
plt.show()

print(f"Mean absolute error on typical LLM weights (σ=0.02):")
print(f"  int4: {int4_err:.6f}")
print(
    f"  NF4:  {nf4_err:.6f}  ({(1 - nf4_err/int4_err)*100:.1f}% less error than int4)"
)
print()
print(
    "→ NF4 reduces quantization error by ~{:.0f}% for normal-distributed weights".format(
        (1 - nf4_err / int4_err) * 100
    )
)
print("  This is why QLoRA uses NF4 instead of regular int4:")
print("  `BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4')`")
print(
    "  (See learning/genai/09-llm-finetuning/02-llm-finetuning-parameter-techniques.ipynb Section 6)"
)

#### #### Your turn — weight distribution width

The NF4 advantage was measured on weights with std = 0.02 (typical LLM initialization). Change `WEIGHT_STD` to `0.5` — much wider. Does NF4 still outperform uniform int4? At what point does NF4's advantage disappear, and why?


In [ ]:
#  Part 6: Your turn — weight distribution width
# # CHANGE: set WEIGHT_STD to 0.001, 0.02 (baseline), 0.1, or 0.5
WEIGHT_STD = 0.02  # # CHANGE: 0.001 = very concentrated near zero, 0.5 = very spread out

# generate a weight sample at the chosen spread for comparison
sample_c = np.random.RandomState(99).randn(1000) * WEIGHT_STD
w_int4_c = quantize_to_levels(sample_c, int4_levels)
w_nf4_c  = quantize_to_levels(sample_c, nf4_quantiles)

# reconstruction error for each scheme at this weight spread
err_int4_c = np.abs(sample_c - w_int4_c).mean()
err_nf4_c  = np.abs(sample_c - w_nf4_c).mean()
# NF4's relative error reduction vs. int4 at this spread
improvement = (1 - err_nf4_c / err_int4_c) * 100 if err_int4_c > 0 else 0.0

print(f"Weight std = {WEIGHT_STD}  (baseline NF4 advantage was ~{(1 - nf4_err/int4_err)*100:.0f}%)")
print(f"  int4 mean abs error: {err_int4_c:.6f}")
print(f"  NF4  mean abs error: {err_nf4_c:.6f}")
print(f"  NF4 improvement:     {improvement:.1f}%")
print()
# classify how much NF4's advantage holds at this weight spread
if improvement > 25:
    print(f"→ NF4 wins by {improvement:.0f}% — weights concentrate near zero (NF4's optimal regime).")
elif improvement > 5:
    print(f"→ NF4 modestly better by {improvement:.0f}% — weights partially fill the tails.")
else:
    print(f"→ NF4 barely helps at std={WEIGHT_STD} — weights spread uniformly (int4's sweet spot).")
print(f"  Takeaway: NF4 is tuned for normally-distributed weights.")
print(f"  If weights ever become uniform (e.g., after aggressive regularization), int4 catches up.")


#### What just happened — and what's missing

NF4's non-uniform spacing is a direct application of information theory: place your 16 levels where probability mass actually lives, not where the axis is uniform. For normal-distributed weights (the vast majority of LLM weights), this reduces reconstruction error by ~30–50% vs. uniform int4 — which is exactly why QLoRA can train competitively with full bf16 fine-tuning despite freezing the base model in 4 bits.

**The bridge to 09-llm-finetuning:** when you called `BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4")` in the fine-tuning notebook, the arithmetic you just measured is what made that call safe to use. Part 7 closes the loop: the full comparison across all six methods, and the final Riverside recommendation.


---

## Part 7 — Summary and Closing Decision

We have worked through six quantization methods, all evaluated against Riverside's constraint: 16 GB MacBook, no internet, manuscript confidentiality. The table below maps what we learned to the original roadmap:

| Part | Concept     | Riverside answer                                          |
| ---- | ----------- | --------------------------------------------------------- |
| 1    | int8 basics | scale/zp rounding error is tiny per-weight; memory halves |
| 2    | Dynamic PTQ | perplexity delta < 0.5 — negligible quality impact        |
| 3    | Static PTQ  | calibration on literary corpus gains another ~0.1–0.3     |
| 4    | GPTQ        | int4 = 3.5 GB for 7B; +0.5–1.5 perplexity (acceptable)    |
| 5    | GGUF Q4_K_M | 4.1 GB, ~30 tok/s on M3, no Python needed                 |
| 6    | NF4         | explains QLoRA's base model storage; bridges to 09-llm-finetuning    |


In [ ]:
#  Closing Decision
print("=" * 60)
print("  CLOSING DECISION — Riverside MacBook Deployment")
print("=" * 60)
print()
print(
    f"  Constraint: MacBook {MACBOOK_VRAM_GB} GB unified memory (must leave ≥2 GB for system)"
)
print()
print("  Options evaluated:")
print(f"    bf16 (baseline): 14.0 GB   OOM  (only 2 GB headroom)")
print(f"    dynamic int8:     7.0 GB    (9 GB free, acceptable perplexity +0.1)")
print(f"    GPTQ int4:        3.5 GB    (12.5 GB free, perplexity +0.5–1.5)")
print(f"    GGUF Q4_K_M:      4.1 GB    (11.9 GB free, ~30 tok/s on M3)")
print()
print("  RECOMMENDATION: GGUF Q4_K_M via llama.cpp")
print("    - No Python environment needed on author MacBooks")
print("    - ~30 tokens/second on Apple M3 (responsive for editing assistant)")
print("    - Q4_K_M quality: +0.3 perplexity over bf16 (barely perceptible)")
print("    - Memory: 4.1 GB → 11.9 GB free for other apps")
print()
print("  ALTERNATIVE for developers: dynamic int8 via PyTorch")
print("    torch.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)")
print("    5 lines of code, 7 GB, +0.1 perplexity")
print()
print("  RULE: Use the lightest quantization that keeps perplexity delta < 1.0")
print("  For literary editing: Q4_K_M stays well within that threshold.")

---

## Toy to Production Bridge

The executable sections use small tensors or GPT-2; AWQ and QAT remain conceptual because faithful demonstrations require production kernels or retraining. Here is how each lesson maps to a real deployment:

| This notebook | Scale | Production equivalent | Production scale |
| --- | --- | --- | --- |
| `quantize_int8` on a 768 x 768 weight tensor | 589K params | Quantize every supported linear layer | Billions of parameters |
| Dynamic PTQ on GPT-2 | 124M parameters | Backend-supported dynamic int8 conversion | 7B: roughly 14 GB to 7 GB of raw weights |
| Static PTQ with five calibration sentences | Five forward passes | Calibrate on representative held-out traffic | Commonly hundreds or thousands of samples |
| GPTQ conceptual walk | One layer | Hessian-guided groupwise conversion | Hours for a full large model |
| AWQ activation-error equation and workflow | Conceptual | Search channel scales and emit an AWQ artifact | Calibration plus supported int4 kernels |
| GGUF format comparison | Reference benchmarks | `llama.cpp` with a versioned GGUF artifact | Native CPU, Metal, or other supported backend |
| QAT fake-quantization and STE derivation | Conceptual | Hardware-targeted retraining followed by integer conversion | Full training/evaluation pipeline |
| NF4 versus int4 sample | 1K synthetic weights | `bitsandbytes` NF4 frozen base with LoRA adapters | 7B base: roughly 3.5 GB raw 4-bit values plus training memory |

At production scale, conversion time, calibration quality, backend kernels, and artifact compatibility become first-class concerns. The small demonstrations teach the controlling arithmetic; they do not substitute for evaluation of the converted artifact on the exact target runtime.

---

## What This Notebook Covered (and What It Did Not)

### Tier 1 - Implemented as Runnable Notebook Experiments

- int8 quantization math: scale, zero point, and rounding error measured on real weights
- Dynamic PTQ: applied to GPT-2 with perplexity measurement
- Static PTQ: calibration flow built around a literary corpus
- GGUF formats: Q4_K_M, Q5_K_M, and Q8_0 compared using reference characteristics
- NF4: non-uniform levels visualized and reconstruction error compared with uniform int4

### Tier 2 - Detailed Conceptual Coverage

- **GPTQ:** Hessian-guided error redistribution, memory impact, tradeoffs, and production tooling
- **AWQ:** activation-amplified error intuition, channel scaling, calibration workflow, pros/cons, and comparison with GPTQ
- **QAT:** fake quantization, straight-through estimation, conversion workflow, pros/cons, and distinction from QLoRA

These methods remain conceptual where a toy implementation would be misleading. GPTQ and AWQ require representative calibration plus supported large-model kernels; QAT requires a real retraining and conversion pipeline.

### Tier 3 - Named Only

- **SpQR:** sparse quantization with individual weight-importance scores
- **SmoothQuant:** shifts activation outlier difficulty into weights before quantization
- **QuIP:** incoherence processing for improved low-bit quantization

---

## When to Use What

| Constraint | Prefer | Why | Main cost |
| --- | --- | --- | --- |
| Simple CPU inference in PyTorch | Dynamic int8 | Minimal setup and no calibration pass | Runtime activation-range overhead |
| Stable CPU workload with representative data | Static int8 PTQ | Calibrated activation ranges reduce runtime work | Calibration quality and backend conversion |
| CUDA server needing weight-only int4 | GPTQ or AWQ | Strong compression with optimized kernels | Calibration plus runtime-specific artifacts |
| End-user deployment without Python | GGUF with `llama.cpp` | Portable native artifact across supported CPU/Metal backends | Format and backend must be benchmarked together |
| PTQ quality is measurably unacceptable | QAT | Learns around simulated quantization noise | Retraining cost and hardware-specific conversion |
| Adapting a much larger model with limited GPU VRAM | QLoRA | NF4 shrinks the frozen base while LoRA keeps trainable state small | CUDA/kernel requirements and slower dequantized computation |
| Maximum quality and memory is available | bf16 or fp32 | Avoids quantization error | Largest memory and bandwidth cost |

Treat this table as a shortlist, not a benchmark result. Convert with the intended toolchain, then evaluate quality, latency, memory, and compatibility on the exact deployment backend.

**Next:** `learning/ai-infrastructure/07-inference-systems/` covers KV cache, continuous batching, speculative decoding, and serving once the model fits.

---

## Key Insights to Keep

- **Quantization error is tiny per weight but composes across billions** — int8 mean absolute error ≈ 0.0002 per weight; the threat is layer-by-layer accumulation, not individual rounding.
- **Dynamic PTQ is 5 lines of code and negligible quality cost** — `quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)` halves model size with < +0.5 perplexity delta for most transformer workloads.
- **Calibration domain matters for static PTQ** — activation ranges estimated on literary text are wrong for technical text; always calibrate on data from your deployment domain.
- **GPTQ makes int4 practical by correcting errors in context** — naïve int4 compounds; GPTQ's Hessian redistribution keeps the layer's total output close to fp32 despite each weight having only 16 levels.
- **GGUF Q4_K_M is the "just works" choice for Apple Silicon** — no Python, no CUDA, ~30 tok/s on M3, +0.3 perplexity vs bf16, fits in a fraction of a 16 GB MacBook's RAM.
- **NF4 wins on normal-distributed weights because it places levels where probability lives** — ~30–50% lower reconstruction error vs. uniform int4 when std ≈ 0.02; the advantage shrinks for wide or uniform distributions.
- **The right question is not "which method" but "which constraint binds"** — deployment environment (PyTorch available?), memory budget, latency target, and quality floor each point to a different method.


---

## Production and Cloud Quantization Release Checklist

A quantized model is a new deployable artifact, not merely a smaller copy of the baseline. Riverside's primary release target is **GGUF Q4_K_M on llama.cpp with Apple Metal**; dynamic/static int8 and GPTQ int4 remain separate artifacts because their kernels, calibration needs, and serving backends differ.

### Calibration data and versioning

- Pin the calibration corpus by dataset ID, immutable version, content SHA-256, preprocessing revision, tokenizer revision, and random seed. Store only approved, de-identified literary samples; never package manuscript text with the model.
- Keep calibration, quality-evaluation, and canary datasets disjoint. Static PTQ and GPTQ use representative calibration data; dynamic int8 and GGUF conversion do not make a calibration set optional if the chosen converter or benchmark workflow requires one.
- Record sample count, sequence-length distribution, language/domain mix, and exclusions. A release must be reproducible from the manifest without relying on a mutable `latest` dataset.

### Hardware and backend compatibility

| Artifact | Intended backend | Required compatibility check |
|---|---|---|
| Dynamic/static int8 | PyTorch CPU | Quantized engine and CPU instruction set (`fbgemm` on x86; test an ARM-supported engine separately) |
| GPTQ int4 | CUDA inference runtime | GPU compute capability, supported kernels, group size, and model architecture |
| GGUF Q4_K_M | llama.cpp | GGUF schema, llama.cpp build/commit, Metal support, context length, and tokenizer metadata |
| NF4 | bitsandbytes for QLoRA training | CUDA, bitsandbytes version, compute dtype, and adapter/base-model pairing |

Do not promote an artifact because it loads on the build host. Test the exact model format, runtime build, accelerator, operating system, and fallback path used in production or in the target cloud instance family.

### Release gates and artifact manifest

Compare every candidate with the pinned bf16/fp32 baseline on the same quality set and serving workload. For this notebook's literary assistant, sensible starting gates are: perplexity delta $\leq 1.0$, generation-regression pass rate $\geq 99\%$, p95 time-to-first-token within the product SLO, sustained throughput $\geq 25$ tokens/s on the target M-series tier, peak model memory $\leq 6$ GB, and no unsupported backend combinations. Tighten these thresholds with production measurements rather than treating the reference benchmark as an SLA.

The artifact manifest should identify the base model and revision, quantization method and parameters, calibration provenance, tokenizer, converter/runtime versions, target hardware, artifact size and SHA-256, evaluation dataset and results, approver, build timestamp, and source commit. Sign and store the manifest beside the immutable artifact.

### Canary, rollback, and observability

Promote through offline gates, shadow traffic, a small canary cohort, and then staged rollout. Keep the previous model and runtime warm enough for a one-step rollback; rollback on quality regressions, load failures, memory pressure, latency/throughput gate violations, or elevated user corrections. A rollback must restore the **artifact, tokenizer, prompt/template, and runtime** as one versioned unit.

Observe p50/p95/p99 time-to-first-token, tokens/s, peak and steady memory, load failures, fallback rate, request truncation, output length, sampled quality scores, user rejection/correction rate, and drift from the calibration/evaluation domain. Tag every event with release ID, artifact digest, method, backend, hardware tier, and runtime version so a mixed fleet does not hide a backend-specific regression.

In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path
import hashlib
import json
import platform

RUN_PRODUCTION_CALIBRATION = False
RUN_PRODUCTION_EXPORT = False
RUN_PRODUCTION_EVALUATION = False
RUN_PRODUCTION_ARTIFACT_VALIDATION = False
RUN_PRODUCTION_ROLLOUT = False


@dataclass(frozen=True)
class ProductionQuantizationConfig:
    release_id: str
    base_model_id: str
    base_model_revision: str
    method: str
    backend: str
    target_hardware: str
    calibration_dataset_id: str
    calibration_dataset_version: str
    calibration_dataset_sha256: str
    calibration_samples: int
    artifact_path: str
    previous_release_id: str
    max_perplexity_delta: float = 1.0
    min_generation_pass_rate: float = 0.99
    max_p95_ttft_ms: float = 750.0
    min_tokens_per_second: float = 25.0
    max_peak_memory_gb: float = 6.0


PRODUCTION_CONFIG = ProductionQuantizationConfig(
    release_id="riverside-llama3-7b-q4km-v1",
    base_model_id="meta-llama/Meta-Llama-3-7B",
    base_model_revision="PIN_MODEL_COMMIT",
    method="gguf_q4_k_m",
    backend="llama.cpp-metal",
    target_hardware="apple-m3-16gb",
    calibration_dataset_id="riverside-literary-calibration",
    calibration_dataset_version="2026-07-30.1",
    calibration_dataset_sha256="PIN_DATASET_SHA256",
    calibration_samples=512,
    artifact_path="artifacts/riverside-llama3-7b-q4_k_m.gguf",
    previous_release_id="riverside-llama3-7b-q4km-v0",
)

BACKEND_COMPATIBILITY = {
    "dynamic_int8": {"pytorch-cpu-fbgemm"},
    "static_int8": {"pytorch-cpu-fbgemm"},
    "gptq_int4": {"cuda-gptq-runtime"},
    "gguf_q4_k_m": {"llama.cpp-metal", "llama.cpp-cpu"},
    "nf4_qlora": {"bitsandbytes-cuda"},
}

assert not any((
    RUN_PRODUCTION_CALIBRATION,
    RUN_PRODUCTION_EXPORT,
    RUN_PRODUCTION_EVALUATION,
    RUN_PRODUCTION_ARTIFACT_VALIDATION,
    RUN_PRODUCTION_ROLLOUT,
)), "Production actions must be explicitly enabled in a reviewed pipeline."

print(json.dumps(asdict(PRODUCTION_CONFIG), indent=2))

In [ ]:
def build_production_plan(config):
    supported_backends = BACKEND_COMPATIBILITY.get(config.method, set())
    compatibility_ok = config.backend in supported_backends
    calibration_required = config.method in {"static_int8", "gptq_int4"}

    return {
        "release_id": config.release_id,
        "compatibility": {
            "status": "PASS" if compatibility_ok else "FAIL",
            "method": config.method,
            "backend": config.backend,
            "allowed_backends": sorted(supported_backends),
            "build_host": platform.platform(),
            "target_hardware": config.target_hardware,
        },
        "stages": [
            {
                "name": "calibration",
                "enabled": RUN_PRODUCTION_CALIBRATION,
                "required": calibration_required,
                "inputs": {
                    "dataset_id": config.calibration_dataset_id,
                    "version": config.calibration_dataset_version,
                    "sha256": config.calibration_dataset_sha256,
                    "samples": config.calibration_samples,
                },
                "action": "Run representative forward passes; never fit ranges on evaluation data.",
            },
            {
                "name": "export",
                "enabled": RUN_PRODUCTION_EXPORT,
                "action": "Convert with a pinned toolchain and emit the immutable GGUF/PTQ/GPTQ artifact.",
            },
            {
                "name": "evaluation",
                "enabled": RUN_PRODUCTION_EVALUATION,
                "action": "Benchmark baseline and candidate on identical quality and serving workloads.",
            },
            {
                "name": "artifact_validation",
                "enabled": RUN_PRODUCTION_ARTIFACT_VALIDATION,
                "action": "Verify format, size, digest, tokenizer metadata, and target-runtime loadability.",
            },
            {
                "name": "rollout",
                "enabled": RUN_PRODUCTION_ROLLOUT,
                "action": "Shadow, canary at 1%, then promote 10/25/50/100% with automatic rollback.",
                "rollback_release_id": config.previous_release_id,
            },
        ],
    }


PRODUCTION_PLAN = build_production_plan(PRODUCTION_CONFIG)
assert PRODUCTION_PLAN["compatibility"]["status"] == "PASS"
print(json.dumps(PRODUCTION_PLAN, indent=2))

In [ ]:
BUILD_METADATA = {
    "tokenizer_revision": "PIN_TOKENIZER_COMMIT",
    "quantizer": "llama.cpp quantize Q4_K_M",
    "quantizer_version": "PIN_LLAMA_CPP_COMMIT",
    "runtime_version": "PIN_LLAMA_CPP_RUNTIME_COMMIT",
    "evaluation_dataset": "riverside-literary-eval@2026-07-30.1",
    "source_commit": "PIN_SOURCE_COMMIT",
    "approver": "PENDING",
    "expected_artifact_sha256": "PIN_ARTIFACT_SHA256",
}

MEASURED_CANDIDATE_METRICS = {
    "perplexity_delta": None,
    "generation_pass_rate": None,
    "p95_ttft_ms": None,
    "tokens_per_second": None,
    "peak_memory_gb": None,
}


def sha256_file(path, chunk_bytes=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as artifact_file:
        for chunk in iter(lambda: artifact_file.read(chunk_bytes), b""):
            digest.update(chunk)
    return digest.hexdigest()


def gate_result(value, threshold, comparison):
    if value is None:
        return {"status": "NOT_RUN", "value": None, "threshold": threshold}
    passed = value <= threshold if comparison == "max" else value >= threshold
    return {"status": "PASS" if passed else "FAIL", "value": value, "threshold": threshold}


def build_release_report(config, metrics):
    artifact_path = Path(config.artifact_path)
    artifact_check = {
        "status": "NOT_RUN",
        "path": str(artifact_path),
        "exists": artifact_path.is_file(),
        "size_bytes": None,
        "sha256": None,
    }

    if RUN_PRODUCTION_ARTIFACT_VALIDATION:
        if not artifact_path.is_file():
            artifact_check["status"] = "FAIL"
        elif BUILD_METADATA["expected_artifact_sha256"].startswith("PIN_"):
            artifact_check["status"] = "BLOCKED_UNPINNED_DIGEST"
        else:
            artifact_check.update(
                status="PASS",
                size_bytes=artifact_path.stat().st_size,
                sha256=sha256_file(artifact_path),
            )
            if artifact_check["sha256"] != BUILD_METADATA["expected_artifact_sha256"]:
                artifact_check["status"] = "FAIL"

    gates = {
        "perplexity_delta": gate_result(metrics["perplexity_delta"], config.max_perplexity_delta, "max"),
        "generation_pass_rate": gate_result(metrics["generation_pass_rate"], config.min_generation_pass_rate, "min"),
        "p95_ttft_ms": gate_result(metrics["p95_ttft_ms"], config.max_p95_ttft_ms, "max"),
        "tokens_per_second": gate_result(metrics["tokens_per_second"], config.min_tokens_per_second, "min"),
        "peak_memory_gb": gate_result(metrics["peak_memory_gb"], config.max_peak_memory_gb, "max"),
    }

    manifest = {
        "schema_version": 1,
        "release": asdict(config),
        "build": BUILD_METADATA,
        "artifact": artifact_check,
        "evaluation": {"metrics": metrics, "gates": gates},
        "rollout": {
            "enabled": RUN_PRODUCTION_ROLLOUT,
            "stages_percent": [0, 1, 10, 25, 50, 100],
            "rollback_release_id": config.previous_release_id,
            "rollback_on_any_gate_failure": True,
        },
    }

    gate_statuses = {result["status"] for result in gates.values()}
    manifest["release_status"] = (
        "FAIL" if "FAIL" in gate_statuses or artifact_check["status"] == "FAIL"
        else "READY_FOR_REVIEW" if gate_statuses == {"PASS"} and artifact_check["status"] == "PASS"
        else "NOT_RUN"
    )
    return manifest


PRODUCTION_REPORT = build_release_report(PRODUCTION_CONFIG, MEASURED_CANDIDATE_METRICS)
assert PRODUCTION_REPORT["release_status"] == "NOT_RUN"
print(json.dumps(PRODUCTION_REPORT, indent=2))